# 01 — Preprocessing

My part of Module 1: turning the raw OSM waterways shapefile into a clean riparian buffer and
river-line layer, for three regions — **Kasarani** (the calibration case study), **Gatharaini**,
and **Motoine**. Covers Steps 1-2 below; Steps 3-4 (Sentinel-2 composite, feature table) are
still open for whoever picks this up next — see the README.

In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box

WATERWAYS_PATH = '../data/vectors/gis_osm_waterways_free_1.shp'
OUT_DIR = '../data/processed'

## Step 1 — Define the three regions

One consistent method for all three: a fixed-radius circle around a center point, not a
hand-picked bounding box. A bounding box is tempting but dangerous here — get the extent even
slightly wrong and you're screening a different population of buildings than whatever
ground-truth count you're calibrating against. One radius for every region also keeps them
comparable to each other.

- **Kasarani** — center matches Pamoja Trust's actual field-survey area.
- **Gatharaini / Motoine** — center is the geometric midpoint of the named river itself (no
  independent ground truth exists for either, so there's nothing else to match against).

In [2]:
CASE_STUDY_RADIUS_KM = 3

REGIONS = {
    'Kasarani':   {'method': 'point', 'center': (36.8969, -1.2296)},
    'Gatharaini': {'method': 'river_name', 'river_name': 'Gatharaini River'},
    'Motoine':    {'method': 'river_name', 'river_name': 'Motoine River'},
}


def get_region_center(waterways, region_key):
    cfg = REGIONS[region_key]
    if cfg['method'] == 'point':
        return cfg['center']
    river = waterways[waterways['name'] == cfg['river_name']]
    if river.empty:
        raise ValueError(f"No waterway named {cfg['river_name']!r} found in {WATERWAYS_PATH}")
    midpoint_metric = river.to_crs(epsg=32737).union_all().centroid
    midpoint = gpd.GeoSeries([midpoint_metric], crs=32737).to_crs(epsg=4326).iloc[0]
    return (midpoint.x, midpoint.y)


def get_region_aoi(center, radius_km=CASE_STUDY_RADIUS_KM):
    """A fixed-radius circle, approximated here as its bounding box just for clipping vectors."""
    lon, lat = center
    pad_deg = radius_km / 111.0
    return box(lon - pad_deg, lat - pad_deg, lon + pad_deg, lat + pad_deg)

## Step 2 — Load and clip the waterways to each region

The shapefile is Kenya-wide (47,157 features) — clip to each region's AOI before doing
anything else with it. Real flowing water only: `fclass` in `river`/`stream`, dropping drains
and canals, which aren't what a riparian buffer policy is about.

In [3]:
def load_waterways():
    return gpd.read_file(WATERWAYS_PATH)


def clip_rivers_to_aoi(waterways, aoi):
    clipped = waterways[waterways.intersects(aoi)].copy()
    rivers_only = clipped[clipped['fclass'].isin(['river', 'stream'])].copy()
    if rivers_only.empty:
        raise ValueError('No river/stream features intersect this AOI — check the AOI bounds.')
    return rivers_only

## Build the riparian buffer

Reproject to a metric CRS (EPSG:32737 — UTM 37S, correct hemisphere for Nairobi) *before*
buffering, since a buffer distance in degrees isn't a reliable metre distance. Buffer, dissolve
overlapping reaches into one shape, then reproject back to EPSG:4326 so it overlays cleanly on
standard web maps.

Also save the clipped river *lines* separately (not just the buffer polygon) in the metric
CRS — whoever calibrates a flagging distance downstream needs true distance-to-river per
building, which a single 60m buffer polygon alone can't give.

In [4]:
def build_riparian_buffer(rivers_only, buffer_m=60):
    rivers_metric = rivers_only.to_crs(epsg=32737)
    buffer_metric = rivers_metric.buffer(buffer_m)
    dissolved = gpd.GeoSeries([buffer_metric.union_all()], crs=32737)
    buffer_global = dissolved.to_crs(epsg=4326)
    return gpd.GeoDataFrame(geometry=buffer_global.explode(index_parts=False).reset_index(drop=True))


def save_river_lines(rivers_only, region_key):
    rivers_metric = rivers_only.to_crs(epsg=32737)[['name', 'fclass', 'geometry']]
    rivers_metric.to_file(f'{OUT_DIR}/{region_key.lower()}_rivers.geojson', driver='GeoJSON')

## Run it for all three regions

Output file names Jane's part depends on: `{region}_riparian_buffer.geojson` and
`{region}_rivers.geojson`.

In [5]:
waterways = load_waterways()
summary = {}

for region_key in REGIONS:
    print(f'--- {region_key} ---')
    center = get_region_center(waterways, region_key)
    aoi = get_region_aoi(center)

    rivers_only = clip_rivers_to_aoi(waterways, aoi)
    buffer_gdf = build_riparian_buffer(rivers_only, buffer_m=60)
    buffer_gdf.to_file(f'{OUT_DIR}/{region_key.lower()}_riparian_buffer.geojson', driver='GeoJSON')
    save_river_lines(rivers_only, region_key)

    summary[region_key] = {
        'river_reaches_in_aoi': len(rivers_only),
        'buffer_polygons': len(buffer_gdf),
    }
    print(summary[region_key])

pd.DataFrame(summary).T

--- Kasarani ---


{'river_reaches_in_aoi': 66, 'buffer_polygons': 4}
--- Gatharaini ---
{'river_reaches_in_aoi': 21, 'buffer_polygons': 2}
--- Motoine ---


{'river_reaches_in_aoi': 41, 'buffer_polygons': 8}


,river_reaches_in_aoi,buffer_polygons
Kasarani,66,4
Gatharaini,21,2
Motoine,41,8


## Handoff notes for whoever does Steps 3-4

- `{region}_riparian_buffer.geojson` and `{region}_rivers.geojson` are ready in
  `data/processed/` for all three regions — both already in the CRS/format the rest of the
  pipeline expects (buffer in EPSG:4326, river lines in EPSG:32737).
- The bbox-vs-circle point from Step 1 matters again downstream: whatever Earth Engine query
  pulls the Sentinel-2 composite should use the *same* `REGIONS`/`CASE_STUDY_RADIUS_KM`
  definitions above, not a separately hand-picked bounding box, or the two halves of this
  notebook will silently disagree about what "Kasarani" even means.
- `save_river_lines()` keeps the river geometry in metric CRS specifically so a later distance-
  to-river calculation doesn't need to reproject on every call.